In [7]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from category_encoders import BinaryEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, OrdinalEncoder, MaxAbsScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score, recall_score, precision_score, make_scorer
from imblearn import FunctionSampler
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_selector as selector
import pandas as pd
from sklearn.model_selection import GroupKFold, StratifiedKFold, StratifiedGroupKFold, KFold, cross_validate
from sklearn import set_config
from sklearn.base import clone
import warnings

warnings.filterwarnings('ignore')

# Esto hace que los steps de las pipelines sean df de pandas por defecto, sino serían np arrays
set_config(transform_output="pandas")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)



In [8]:
df_claim_reviews = pd.read_csv("C:\\Users\\Miguel Fernández\\Desktop\\2º CURSO\\SEGUNDO SEMESTRE\\Aprendizaje Automático\\Trabajo_FInal\\CSV\\claim_reviews.csv")

In [9]:
df_claim_reviews

,review_id,claim_id,review_round,review_datetime,reviewer_id,review_type,auto_risk_score,triage_result,reviewer_notes,perito_id,perito_physical_inspection,damage_consistency_rating,documentation_completeness_pct,recommended_action,days_since_claim
0,REV_00002_1,CLM_00002,1,2025-11-29 14:33:00,R-2512,Initial Triage,16.1,Green - Proceed,"Standard claim, within normal parameters.",NaN,NaN,NaN,NaN,Proceed,1
1,REV_00003_1,CLM_00003,1,2024-06-26 04:36:00,R-1744,Initial Triage,36.3,Yellow - Standard Review,Claimant was vague about accident circumstances.,NaN,NaN,NaN,NaN,Standard Review,2
2,REV_00003_2,CLM_00003,2,2024-07-29 21:36:00,R-1744,Expert Assessment,NaN,NaN,No anomalies found during review.,P-5992,No,2.2,60.2,Recommend Denial,35
3,REV_00003_3,CLM_00003,3,2024-08-15 13:36:00,SIU-209,SIU Investigation,NaN,NaN,Full investigation completed. Findings documen...,P-5992,Yes,NaN,NaN,Claim Denied,51
4,REV_00004_1,CLM_00004,1,2025-08-05 19:55:00,R-3644,Initial Triage,29.8,Green - Proceed,"All documents verified, consistent narrative.",NaN,NaN,NaN,NaN,Proceed,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20888,REV_15413_1,CLM_15413,1,2023-07-06 08:20:00,R-9461,Initial Triage,26.8,Green - Proceed,"Routine processing, no flags raised.",NaN,NaN,NaN,NaN,Proceed,1
20889,REV_15414_1,CLM_15414,1,2025-10-10 01:53:00,R-6221,Initial Triage,67.4,Orange - Escalated Review,Third party account differs slightly from clai...,NaN,NaN,NaN,NaN,Escalated Review,1
20890,REV_15414_2,CLM_15414,2,2025-10-16 13:53:00,R-6221,Expert Assessment,NaN,NaN,Documentation supports claim as reported.,P-8545,Yes,4.3,26.5,Approve with Conditions,7
20891,REV_15417_1,CLM_15417,1,2022-11-10 21:46:00,R-8044,Initial Triage,28.3,Green - Proceed,"Documentation complete, no issues detected.",NaN,NaN,NaN,NaN,Proceed,2


In [11]:
#Ver nulos

df_claim_reviews.isnull().sum()

review_id                             0
claim_id                              0
review_round                          0
review_datetime                       0
reviewer_id                           0
review_type                           0
auto_risk_score                    8626
triage_result                      8626
reviewer_notes                        0
perito_id                         12267
perito_physical_inspection        12267
damage_consistency_rating         13897
documentation_completeness_pct    13897
recommended_action                    0
days_since_claim                      0
dtype: int64

In [17]:
df_claim_reviews.describe(include='all')

,review_id,claim_id,review_round,review_datetime,reviewer_id,review_type,auto_risk_score,triage_result,reviewer_notes,perito_id,perito_physical_inspection,damage_consistency_rating,documentation_completeness_pct,recommended_action,days_since_claim
count,20893,20893,20893.000000,20893,20893,20893,12267.000000,12267,20893,8626,8626,6996.000000,6996.000000,20893,20893.000000
unique,20893,12267,NaN,20784,791,3,NaN,4,40,59,2,NaN,NaN,15,NaN
top,REV_15419_1,CLM_00008,NaN,2025-02-10 08:23:00,R-2207,Initial Triage,NaN,Green - Proceed,"Documentation complete, no issues detected.",P-9976,No,NaN,NaN,Proceed,NaN
freq,1,3,NaN,3,603,12267,NaN,6183,2391,309,4756,NaN,NaN,6183,NaN
mean,NaN,NaN,1.490882,NaN,NaN,NaN,30.992574,NaN,NaN,NaN,NaN,3.606718,80.303902,NaN,7.622218
std,NaN,NaN,0.637157,NaN,NaN,NaN,20.705931,NaN,NaN,NaN,NaN,0.892093,13.696789,NaN,9.188407
min,NaN,NaN,1.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,1.000000,14.700000,NaN,1.000000
25%,NaN,NaN,1.000000,NaN,NaN,NaN,15.200000,NaN,NaN,NaN,NaN,3.000000,71.575000,NaN,2.000000
50%,NaN,NaN,1.000000,NaN,NaN,NaN,29.700000,NaN,NaN,NaN,NaN,3.700000,81.050000,NaN,4.000000
75%,NaN,NaN,2.000000,NaN,NaN,NaN,45.400000,NaN,NaN,NaN,NaN,4.300000,90.700000,NaN,10.000000


CALCULO DEL TARGET

In [ ]:


# Condición A: ronda 3 con resolución negativa SIU
cond_a_actions = {
    "Claim Denied",
    "Claim Withdrawn by Claimant",
    "Pending Litigation",
}
cond_a_claims = set(
    df_claim_reviews.loc[
        (df_claim_reviews["review_round"] == 3)
        & (df_claim_reviews["recommended_action"].isin(cond_a_actions)),
        "claim_id",
    ]
)

# Condición B: ronda 2 con acción desfavorable + inconsistencia de daños < 3.0
cond_b_actions = {
    "Recommend Denial",
    "Refer to Special Investigations Unit",
}
cond_b_claims = set(
    df_claim_reviews.loc[
        (df_claim_reviews["review_round"] == 2)
        & (df_claim_reviews["recommended_action"].isin(cond_b_actions))
        & (df_claim_reviews["damage_consistency_rating"] < 3.0),
        "claim_id",
    ]
)

# Condición C: ronda 1 roja + ronda 2 con inspección física y docs < 65
r1_red_claims = set(
    df_claim_reviews.loc[
        (df_claim_reviews["review_round"] == 1)
        & (df_claim_reviews["triage_result"] == "Red - Full Investigation"),
        "claim_id",
    ]
)
r2_severe_claims = set(
    df_claim_reviews.loc[
        (df_claim_reviews["review_round"] == 2)
        & (df_claim_reviews["perito_physical_inspection"] == "Yes")
        & (df_claim_reviews["documentation_completeness_pct"] < 65),
        "claim_id",
    ]
)
cond_c_claims = r1_red_claims & r2_severe_claims

# Unión lógica A OR B OR C
fraud_claims = cond_a_claims | cond_b_claims | cond_c_claims

# Target final por siniestro
df_target = (
    df_claim_reviews[["claim_id"]]
    .drop_duplicates()
    .assign(fraud_flag=lambda d: d["claim_id"].isin(fraud_claims).astype(int))
    .sort_values("claim_id")
    .reset_index(drop=True)
)

print("Nº siniestros:", len(df_target))
print("Fraudes (fraud_flag=1):", int(df_target["fraud_flag"].sum()))
print("No fraudes (fraud_flag=0):", int((df_target["fraud_flag"] == 0).sum()))

df_target.head(10)

Nº siniestros: 12267
Fraudes (fraud_flag=1): 878
No fraudes (fraud_flag=0): 11389


,claim_id,fraud_flag
0,CLM_00002,0
1,CLM_00003,1
2,CLM_00004,0
3,CLM_00005,0
4,CLM_00006,0
5,CLM_00007,0
6,CLM_00008,1
7,CLM_00009,0
8,CLM_00010,0
9,CLM_00012,0


In [15]:
df_target['fraud_flag'].value_counts(normalize=True)

fraud_flag
0    0.928426
1    0.071574
Name: proportion, dtype: float64

In [16]:
df_target

,claim_id,fraud_flag
0,CLM_00002,0
1,CLM_00003,1
2,CLM_00004,0
3,CLM_00005,0
4,CLM_00006,0
...,...,...
12262,CLM_15412,0
12263,CLM_15413,0
12264,CLM_15414,0
12265,CLM_15417,0


### Teniendo ya calculado el target, vamos a abandonar la tabla claim_reviews (en el readme.md se especifica que solo sirve para calcular el target)

In [20]:
df_claims = pd.read_csv("C:\\Users\\Miguel Fernández\\Desktop\\2º CURSO\\SEGUNDO SEMESTRE\\Aprendizaje Automático\\Trabajo_FInal\\CSV\\claims.csv")

In [21]:
df_claims

,claim_id,policy_id,customer_id,vehicle_id,agent_id,accident_datetime,claim_datetime,fault,accident_area,accident_description,accident_latitude,accident_longitude,police_report_filed,witness_present,number_of_supplements,claimed_amount_eur,repair_workshop
0,CLM_00002,POL_07423,CUS_01111,VEH_01155,AGT_00118,2025-11-02 16:33:00,2025-11-28 06:33:00,Policy Holder,Urban,flooding damage,41.104921,-1.472463,Yes,Unknown,7,1198.91,Garage Ramírez - Murcia
1,CLM_00003,POL_01086,CUS_01750,VEH_00648,AGT_00027,2024-05-12 22:36:00,2024-06-23 15:36:00,Policy Holder,Urban,glass breakage,37.332902,-1.063439,No,Yes,7,11848.24,Carrocerías Díaz - Coruña
2,CLM_00004,POL_06373,CUS_06806,VEH_00899,AGT_00051,2025-07-09 22:55:00,2025-08-03 02:55:00,Policy Holder,Urban,bicycle involved,36.944930,-6.714446,Yes,No,2,1116.78,Carrocerías Díaz - Vigo
3,CLM_00005,POL_04543,CUS_04165,VEH_05968,AGT_00062,2023-11-08 20:14:00,2023-11-20 17:14:00,Policy Holder,Urban,multi-vehicle pileup,42.466545,2.320600,Yes,Yes,1,2246.88,Carrocerías Ruiz - Valencia
4,CLM_00006,POL_04676,CUS_07070,VEH_06082,AGT_00076,2025-08-21 12:39:00,2025-09-30 18:39:00,Policy Holder,Urban,falling object,39.341821,-2.414222,No,Yes,6,449.41,Carrocerías Gómez - Valencia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12262,CLM_15412,POL_07393,CUS_02209,VEH_07329,AGT_00123,2024-12-17 11:01:00,2024-12-26 16:01:00,Policy Holder,Suburban,pedestrian involved,39.549568,0.227634,Yes,No,0,960.23,Mecánica Rubio - Málaga
12263,CLM_15413,POL_00752,CUS_05111,VEH_02884,AGT_00105,2023-06-24 02:20:00,2023-07-04 19:20:00,Policy Holder,Urban,theft attempt,38.652884,-3.561560,No,No,2,1843.21,Taller Domínguez - Alicante
12264,CLM_15414,POL_04232,CUS_01224,VEH_03091,AGT_00042,2025-09-10 06:53:00,2025-10-08 17:53:00,Policy Holder,Urban,theft attempt,38.648536,-6.392743,No,Yes,2,957.81,Taller Rivera - Alicante
12265,CLM_15417,POL_02434,CUS_01106,VEH_06088,AGT_00173,2022-10-11 14:46:00,2022-11-08 04:46:00,Policy Holder,Urban,weather-related,42.800512,-3.779887,No,No,5,2349.20,Chapa y Pintura Castillo - Las Palmas


In [23]:
df_claims.isnull().sum()

claim_id                 0
policy_id                0
customer_id              0
vehicle_id               0
agent_id                 0
accident_datetime        0
claim_datetime           0
fault                    0
accident_area            0
accident_description     0
accident_latitude        0
accident_longitude       0
police_report_filed      0
witness_present          0
number_of_supplements    0
claimed_amount_eur       0
repair_workshop          0
dtype: int64

In [22]:
df_claims.describe(include='all')

,claim_id,policy_id,customer_id,vehicle_id,agent_id,accident_datetime,claim_datetime,fault,accident_area,accident_description,accident_latitude,accident_longitude,police_report_filed,witness_present,number_of_supplements,claimed_amount_eur,repair_workshop
count,12267,12267,12267,12267,12267,12267,12267,12267,12267,12267,12267.000000,12267.000000,12267,12267,12267.000000,12267.000000,12267
unique,12267,6167,3759,6594,180,12227,12228,2,5,20,NaN,NaN,3,3,NaN,NaN,387
top,CLM_15419,POL_00734,CUS_00250,VEH_08192,AGT_00016,2025-05-31 09:23:00,2024-01-11 16:56:00,Policy Holder,Urban,fire damage,NaN,NaN,No,No,NaN,NaN,Carrocerías Ramírez - Vitoria
freq,1,7,19,7,96,2,2,8017,4879,654,NaN,NaN,6684,7194,NaN,NaN,80
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.850914,-2.970615,NaN,NaN,3.477541,2999.131380,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.252370,3.624874,NaN,NaN,2.279481,3873.780393,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.000737,-9.299434,NaN,NaN,0.000000,47.330000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.889615,-6.085252,NaN,NaN,1.000000,912.225000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.833320,-2.989663,NaN,NaN,3.000000,1822.520000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.788675,0.189406,NaN,NaN,5.000000,3555.105000,NaN
